# Proyecto 3 – Mejorando algoritmos con CUDA

Computación Paralela y Distribuida, UVG. Base: **Laboratorio 3 de Data Science (reconocimiento del alfabeto ASL)**.

Este notebook reúne las seis etapas del proyecto. El código está en `src/` y aquí solo se llama:

| Fase | Hardware | Código |
|------|----------|--------|
| 1. Preparación de datos (lectura, decodificación, resize, normalización, split) | **CPU**: secuencial (`src/preprocessing.py`, original) y multiproceso (`src/preprocessing_parallel.py`) | `ProcessPoolExecutor` |
| 2. Entrenamiento e inferencia de la CNN 2 | **GPU**: PyTorch + CUDA + cuDNN (`src/train.py`, `src/model.py`) | `torch.cuda` |
| 3. Kernels propios de la red (conv, pooling, densa) | **GPU**: CUDA Python con Numba (`src/cuda_kernels.py`) | `@cuda.jit` |

`src/benchmark.py` mide todo y `src/monitor.py` registra el uso de cada núcleo de CPU y de la GPU durante cada medición.

> **Antes de abrir este notebook** correr desde la raíz del repo `python src/benchmark.py` (ver README). Aquí se leen los CSV que deja.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)                       # los módulos de src/ usan rutas relativas a la raíz
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.precision", 3)
print("Raíz del repo:", ROOT)
pd.read_csv("results/sistema.csv", index_col=0)

## 1. Análisis de la solución secuencial existente

### Partes principales del programa (Lab 3)

| Parte | Archivo | Función |
|-------|---------|---------|
| Descarga del dataset | `src/dataset.py` | Descarga *ASL Alphabet* de Kaggle con `kagglehub` (87 000 JPG de 200×200) y deja una carpeta por clase. |
| Manifiesto | `src/preprocessing.py::build_manifest` | Recorre las 29 carpetas y toma 700 rutas por clase al azar (seed 42): 20 300 imágenes. |
| Split estratificado | `stratified_split` | 70/15/15 manteniendo la proporción de clases: 14 210 / 3 045 / 3 045. |
| Carga y preprocesamiento | `load_and_preprocess` | Por cada imagen: abre el archivo, decodifica el JPEG, convierte a RGB, redimensiona a 64×64 y divide entre 255. Un ciclo `for` secuencial. |
| Guardado | `build_processed_splits` | Guarda `X_*.npy` y `y_*.npy` en `data/processed/`. |
| Modelos | `notebooks/02_modelos_cnn.ipynb` | CNN 1, CNN 2 y red densa en Keras; la mejor (CNN 2) queda en `models/best_cnn.keras`. |
| Entrenamiento | `model.fit` | 15 épocas, batch 32, Adam (lr 5e-4), early stopping (paciencia 3). |
| Evaluación | `model.predict` + sklearn | Accuracy, precision y recall macro y matriz de confusión sobre test. |

### Descomposición en operaciones (de la imagen al resultado)

| No. | Operación identificada | Descripción de su funcionamiento |
|----:|------------------------|----------------------------------|
| 1 | Listado de archivos | `Path.glob("*.jpg")` en cada una de las 29 carpetas. |
| 2 | Muestreo | `rng.choice` elige 700 rutas por clase sin reemplazo. |
| 3 | Split estratificado | Dos llamadas a `train_test_split` con `stratify=label`. |
| 4 | Lectura del archivo | `Image.open` lee el JPEG del disco. |
| 5 | Decodificación JPEG | `convert("RGB")` fuerza la decodificación (Huffman + IDCT) a 200×200×3. |
| 6 | Redimensionamiento | `resize((64, 64))` con interpolación bilineal: 200×200 → 64×64. |
| 7 | Normalización | `np.asarray(...) / 255.0` a `float32` en [0, 1]. |
| 8 | Codificación de etiqueta | `class_to_idx[label]` → entero 0–28. |
| 9 | Guardado / carga de `.npy` | `np.save` / `np.load` de los arrays (~1 GB en float32). |
| 10 | Convolución 3×3 + ReLU | 3 capas (32, 64, 128 filtros): cada píxel de salida es una suma de 9·C productos. |
| 11 | Batch normalization | Normaliza cada canal con la media y varianza del batch (entrenamiento) o acumuladas (inferencia). |
| 12 | Max pooling 2×2 | Máximo de cada ventana 2×2; reduce a la mitad alto y ancho. |
| 13 | Capas densas | Multiplicación de matrices 4608→256→128→29 + ReLU, con dropout en entrenamiento. |
| 14 | Softmax + entropía cruzada | Pérdida de la predicción frente a la etiqueta. |
| 15 | Backpropagation | Gradiente de la pérdida respecto a los 1.31 M de parámetros, capa por capa. |
| 16 | Actualización Adam | Actualiza cada parámetro con sus momentos de primer y segundo orden. |
| 17 | Validación por época | Forward sobre las 3 045 imágenes de validación. |
| 18 | Inferencia y métricas | Forward sobre test, `argmax` y métricas de sklearn. |

### Operaciones repetitivas

In [ ]:
from model import CNN2, count_parameters

N_TRAIN, N_VAL, N_TEST, BATCH, EPOCHS = 14210, 3045, 3045, 32, 15
batches = -(-N_TRAIN // BATCH)

# multiplicaciones-suma (MAC) por imagen en la pasada hacia adelante
mac = {
    "conv1 (3→32)":   62 * 62 * 32 * 3 * 9,
    "conv2 (32→64)":  29 * 29 * 64 * 32 * 9,
    "conv3 (64→128)": 12 * 12 * 128 * 64 * 9,
    "dense 4608→256": 4608 * 256,
    "dense 256→128":  256 * 128,
    "dense 128→29":   128 * 29,
}
mac_img = sum(mac.values())

rep = pd.DataFrame([
    ("Lectura + decodificación + resize + normalización", "1 JPEG 200×200×3 → 64×64×3 float32", f"{N_TRAIN + N_VAL + N_TEST:,} imágenes"),
    ("Convolución 3×3 (forward)", "9·C MAC por píxel de salida", f"{(62*62*32 + 29*29*64 + 12*12*128):,} píxeles por imagen"),
    ("Max pooling 2×2", "4 valores → 1", f"{(31*31*32 + 14*14*64 + 6*6*128):,} salidas por imagen"),
    ("Capas densas (forward)", "matriz batch×4608 por 4608×256, etc.", f"{(4608*256 + 256*128 + 128*29):,} MAC por imagen"),
    ("Paso de entrenamiento (forward + backward + Adam)", "batch de 32 imágenes", f"{batches} por época × {EPOCHS} = {batches*EPOCHS:,}"),
    ("Actualización Adam", f"{count_parameters(CNN2()):,} parámetros", f"{batches*EPOCHS:,} veces"),
    ("Validación", f"{N_VAL} imágenes", f"{EPOCHS} veces (una por época)"),
], columns=["Operación repetitiva", "Datos que procesa", "Cantidad de repeticiones"])
display(rep)

print(f"MAC por imagen (forward): {mac_img/1e6:.1f} M")
print(f"Entrenamiento ≈ 3 × forward (forward + 2 de backward): "
      f"{3 * mac_img * N_TRAIN * EPOCHS / 1e12:.1f} T MAC en 15 épocas")
pd.Series(mac, name="MAC por imagen").to_frame().assign(porcentaje=lambda d: 100 * d.iloc[:, 0] / mac_img)

Las convoluciones son ~96 % del cómputo de la red y la capa densa 4608→256 tiene ~90 % de los parámetros. Todo ese trabajo son sumas de productos **independientes entre sí** (cada píxel de salida, cada imagen del batch, cada neurona), que es justo lo que una GPU hace bien.

## 2. Medición del tiempo de ejecución secuencial

Procedimiento (`src/benchmark.py`):

1. Misma máquina, mismo conjunto (20 300 imágenes, seed 42), sin otros programas pesados abiertos y con el equipo conectado a la corriente.
2. Una pasada de **calentamiento** antes de medir: la primera lectura del dataset viene del disco y las demás del caché del sistema operativo; sin ella la primera configuración medida queda castigada.
3. **3 repeticiones** de cada configuración, alternadas (secuencial, paralela, secuencial, …) para que cualquier cambio de temperatura o de carga del sistema afecte a ambas por igual. Se reporta promedio y desviación estándar.
4. Cronómetro `time.perf_counter()`. En GPU se llama `torch.cuda.synchronize()` antes de leerlo, porque los kernels se lanzan de forma asíncrona.
5. El entrenamiento del benchmark usa **3 épocas fijas** (sin early stopping) para que todas las configuraciones hagan exactamente el mismo trabajo; el tiempo por época permite extrapolar a las 15 del Lab 3.

La versión secuencial de referencia es el código original: `load_and_preprocess` (un ciclo `for`) y el entrenamiento en CPU con **1 hilo**. También se mide la CPU con todos sus hilos para separar cuánto aporta el paralelismo de la CPU y cuánto la GPU.

In [ ]:
t_pre = pd.read_csv("results/tiempos_preprocesamiento.csv")
t_pre[["config", "rep", "procesos", "total_s", "manifiesto_split_s", "carga_imagenes_s", "cpu_promedio_%", "nucleos_sobre_50%"]]

In [ ]:
t_ent = pd.read_csv("results/tiempos_entrenamiento.csv")
cols = ["config", "rep", "threads", "epochs", "h2d_s", "entrenamiento_s", "s_por_epoca", "inferencia_s", "d2h_s",
        "test_accuracy", "cpu_promedio_%", "gpu_util_promedio_%", "nvidia_smi_mem_max_mb"]
t_ent[[c for c in cols if c in t_ent]]

Factores que influyen en el rendimiento secuencial:

- **Disco y caché del sistema de archivos**: la primera lectura de 20 300 archivos es mucho más lenta que las siguientes.
- **Decodificación JPEG**: es el paso más caro del preprocesamiento y en el código original usa un solo núcleo.
- **Hilos de la librería matemática**: PyTorch en CPU puede usar 1 o todos los núcleos; con 1 hilo el entrenamiento es puramente secuencial.
- **Frecuencia del CPU**: en una laptop baja con la temperatura y si no está conectada a la corriente.
- **Tamaño del batch**: batches pequeños (32) aprovechan peor tanto la CPU como la GPU.

## 3. Identificación de oportunidades de paralelización

| Operación | ¿Paralelizable? | Dónde | Justificación |
|-----------|-----------------|-------|---------------|
| Listado, muestreo y split (1–3) | No vale la pena | CPU, 1 hilo | Solo manipula 20 300 rutas; tarda fracciones de segundo. |
| Lectura + decodificación + resize + normalización (4–8) | **Sí, por datos** | **CPU, varios procesos** | Cada imagen es independiente. Es E/S de archivos y decodificación con mucha lógica de control: trabajo de CPU, no de GPU. Se usan procesos y no hilos por el GIL de Python. |
| Convolución, pooling, densas, BN, softmax (10–14) | **Sí, masivamente** | **GPU (CUDA)** | Millones de sumas de productos independientes por batch: paralelismo de datos de grano fino, ideal para miles de hilos. |
| Backpropagation (15) | **Sí** | **GPU** | Son las mismas operaciones (convoluciones y productos de matrices) en orden inverso. |
| Adam (16) | **Sí** | **GPU** | Operación elemento a elemento sobre 1.31 M de parámetros. |
| Épocas y batches entre sí | No | — | Cada paso depende de los pesos que dejó el anterior (dependencia secuencial del descenso de gradiente). |

Conclusión: el paralelismo de **la CPU** se aplica a los **archivos** (grano grueso: una imagen por tarea) y el de **la GPU** a la **matemática de la red** (grano fino: un hilo por píxel o neurona).

## 4. Diseño del algoritmo paralelo

### Fase 1: CPU multiproceso
Las 20 300 rutas se dividen en `4 × núcleos` bloques contiguos y un `ProcessPoolExecutor` reparte los bloques entre procesos (balanceo dinámico: un proceso que termina antes toma otro bloque). `map` conserva el orden, así que el resultado es **idéntico bit a bit** al secuencial (se verifica en cada repetición).

### Fase 2: GPU con PyTorch/cuDNN
- Los tres splits se copian **una sola vez** a la GPU (~1 GB) y se quedan ahí: no hay copias por batch.
- El barajado de cada época se hace con `torch.randperm` dentro de la GPU.
- Cada operación de la red es un kernel de cuDNN/cuBLAS; `cudnn.benchmark=True` elige el algoritmo de convolución más rápido para estos tamaños.
- Variante con precisión mixta (AMP, FP16).

### Fase 3: kernels CUDA propios (Numba)
Para mostrar explícitamente el reparto en hilos, bloques y grid se implementó a mano la pasada hacia adelante de la CNN 2 (configuración para un lote de 256 imágenes):

In [ ]:
import cuda_kernels as ck
pd.DataFrame(ck.launch_config_table())

- **conv3x3 + ReLU + BN (fusionado)**: 1 hilo = 1 píxel de salida `(n, f, y, x)`. Bloque de 16×16 hilos = parche 16×16 de un mapa. Grid `(⌈W/16⌉, ⌈H/16⌉, N·F)`. Los 9·C pesos del filtro se copian a **memoria compartida** una vez por bloque.
- **max pooling 2×2**: 1 hilo por píxel de salida, mismo esquema.
- **densa**: multiplicación de matrices por **mosaicos (tiling) de 16×16 en memoria compartida**; cada dato de memoria global se reutiliza 16 veces.
- Se procesan lotes de 256 imágenes para que los mapas intermedios quepan en una GPU de 4 GB.

## 5. Implementación con CUDA Python y medición

Implementación en `src/train.py` (PyTorch CUDA) y `src/cuda_kernels.py` (Numba CUDA). Mismos datos, mismas épocas y mismo batch que la versión secuencial; el tiempo de GPU incluye por separado las transferencias CPU→GPU (`h2d`) y GPU→CPU (`d2h`).

In [ ]:
resumen = pd.read_csv("results/resumen_speedup.csv")
resumen

In [ ]:
for fase in resumen["fase"].unique():
    display(Image(f"images/p3_speedup_{fase}.png"))

In [ ]:
t_k = pd.read_csv("results/tiempos_kernels.csv")
cols = ["total_s", "h2d_s", "kernels_s", "d2h_s", "error_max_vs_pytorch", "test_accuracy"]
t_k.groupby("config", sort=False)[[c for c in cols if c in t_k]].mean()

### Evidencia de uso de recursos
Uso de cada núcleo lógico de la CPU (mapa de calor) y de la GPU (uso y memoria) durante cada configuración, registrado cada 0.5 s (`results/logs/`). Se complementa con las capturas del Administrador de Tareas y `nvidia-smi` en `docs/report/capturas/`.

In [ ]:
for png in sorted(Path("images").glob("p3_recursos_*.png")):
    print(png.name)
    display(Image(png))

In [ ]:
final = pd.read_csv("results/metricas_modelo_final.csv")
hist = pd.read_csv("results/historial_modelo_final.csv")
ax = hist.plot(x="epoch", y=["accuracy", "val_accuracy"], marker="o", figsize=(7, 4), title="CNN 2 entrenada en GPU")
ax.set_ylabel("accuracy")
final[["device", "epochs", "entrenamiento_s", "s_por_epoca", "test_accuracy", "test_precision", "test_recall"]]

## 6. Análisis y conclusiones

La discusión completa (speedup por fase, causas de las diferencias y factores que justifican o limitan CUDA) está en `docs/report/informe_proyecto3.md`.